In [ ]:
from ipyfilechooser import FileChooser
from interface_jupyter import Interface
import ipywidgets as widgets
import subprocess, sys, threading, os


class PeakPrecompressorUI(Interface):
    def __init__(self):
        super().__init__(supported_extensions=[".txt"])
        self._setup_default_parameters()
        self._create_parameter_widgets()
        self._create_base_widgets()
        self._create_action_widgets()
        self._create_precompress_widget()
        self._setup_callbacks()
        self._setup_environment()

    def _setup_default_parameters(self):
        self.rt1_penalty = "1"
        self.rt2_penalty = "5"
        self.similarity_cutoff = "95"
        self.num_cores = "1"
        self.common_ions = None
        self.quant_method = "T"
        self.output_files = False
    
    def _create_parameter_widgets(self):
        self.w_rt1_penalty = widgets.Text(value=self.rt1_penalty)
        self.rt1_penalty = self._bold_widget("RT1 Penalty", self.w_rt1_penalty)
        self.rt1_penalty_def = self.create_help_text(
            "Penalty used for first retention time errors.  Defaults to 1."
        )
        self.w_rt2_penalty = widgets.Text(value=self.rt2_penalty)
        self.rt2_penalty = self._bold_widget("RT2 Penalty", self.w_rt2_penalty)
        self.rt2_penalty_def = self.create_help_text(
            "Penalty used for second retention time errors. Defaults to 5."
        )
        self.w_similarity_cutoff = widgets.Text(value=self.similarity_cutoff)
        self.similarity_cutoff = self._bold_widget("Similarity Cutoff", self.w_similarity_cutoff)
        self.similarity_cutoff_def = self.create_help_text(
            "Adjusts peak similarity threshold required for alignment."
            "Adjust in concordance with RT1 and RT2 penalties. " \
            "Will be ignored if autoTuneMatchStrigency is TRUE. Defaults to 90."
            )
        self.w_num_cores = widgets.Text(value=self.num_cores)
        self.num_cores = self._bold_widget("Number of Cores", self.w_num_cores)
        self.num_cores_def = self.create_help_text(
            "Number of CPU cores to use for processing. Defaults to 1."
        )

    def _create_precompress_widget(self):
        self.txt_title = widgets.HTML(value="<H1>Peak precompressor</H1>")
        self.run_button, self.stop_button, self.clear_button, self.output = self._create_action_widgets()

    def _on_button_click(self, b):
        with self.output:
            self.output.clear_output()
            print("Starting peak precompression...")
            if not hasattr(self, 'output_chooser') or not self.output_chooser.selected_path:
                print("Output directory cannot be empty")
                return
            print("\n Collecting files from selections...   ")
            selected_files = self.get_all_files_from_selections()
            if not selected_files:
                print("Please select files or folders containing .txt files.")
                return
            print(f"\n✅ {len(selected_files)} compatible files found")
            for i, f in enumerate(selected_files, 1):
                print(f"  {i}. {selected_files [i-1]}")
            print(f"{'='*60}")

            self._start_subprocess_precompressor(selected_files)

    def _start_subprocess_precompressor(self, selected_files):
            # Implement the logic to start the subprocess for peak precompression
        precompress_params = [
            sys.executable,
            '/app/src/peak_precompressor_cli.py',
            '--rt1_penalty', self.w_rt1_penalty.value,
            '--rt2_penalty', self.w_rt2_penalty.value,
            '--similarity_cutoff', self.w_similarity_cutoff.value,
            '--num_cores', self.w_num_cores.value,
            '--common_ions', self.w_common_ions.value,
            '--quant_method', self.w_quant_method.value,
            '--output_dir', self.get_output_path(),
            
        ]
        # cas des booleens
        if self.output_files:
            precompress_params.append('--output')
        # cas des listes
        precompress_params +=["--input"] + selected_files

        precompress_params= list(map(str, precompress_params))

        self.current_process = subprocess.Popen(
            precompress_params,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        self.stop_button.disabled = False 

        def stream_output(proc, output_widget):
            try:
                for line in iter(proc.stdout.readline, ''):
                    with output_widget:
                        print(line, end='')
                    # Vérifie si le process est terminé après chaque ligne
                    if proc.poll() is not None:
                        break
            except Exception as e:
                with output_widget:
                    print(f"⚠️ Error reading process output: {e}")
            finally:
                proc.stdout.close()
                retcode = proc.wait()
                with output_widget:
                    if retcode == 0:
                        print("\n✅ Analyse terminée avec succès")
                    else:
                        print(f"\n❌ L'analyse a échoué avec le code de retour {retcode}")
                        print(f"\n{'='*60}")
                self.stop_button.disabled = True
                self.current_process = None

        threading.Thread(
            target=stream_output,
            args=(self.current_process,
                  self.output),
                  daemon=True
            ).start()
        
    def display(self):
        display(
            self.txt_title,
            widgets.VBox([self._vbox, self._vbox2]),
            self.rt1_penalty,
            self.rt1_penalty_def,
            self.rt2_penalty,
            self.rt2_penalty_def,
            self.similarity_cutoff,
            self.similarity_cutoff_def,
            self.num_cores,
            self.num_cores_def,

            widgets.HBox([self.run_button, self.stop_button, self.clear_button]),
        self.output
        )

In [ ]:
precompress_ui = PeakPrecompressorUI()
precompress_ui.display()